# Script to run GPT using the batch API

In [1]:
from openai import OpenAI

from scripts.prompting.jup_utils import open_input_file, from_dataset_to_batch_req, upload_batch_request, run_batch, check_batch_status, convert_response

Step-0: Initialize the client

In [47]:
with open('/Users/aloneirew/workspace/DenseEREGraph/data/my_data/api_key.txt', 'r') as file_api:
    api_key = file_api.readline().strip()

_client = OpenAI(api_key=api_key)

# _test_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/EventFullTrainExports/test'
# _train_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/EventFullTrainExports/train'
# _dot_train_data = open_input_file('/Users/aloneirew/workspace/DenseEREGraph/data/DOT_format/EventFull_dev_dot.json')
# _test_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/MATRES/in_my_format_all_pairs/test'
# _train_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/MATRES/in_my_format/train'
# _dot_train_data = open_input_file('/Users/aloneirew/workspace/DenseEREGraph/data/DOT_format/MATRES_train_dot.json')
_test_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/NarrativeTime/converted_no_overlap/test_18ment'
_train_folder = '/Users/aloneirew/workspace/DenseEREGraph/data/NarrativeTime/converted_no_overlap/test_18ment'
_dot_train_data = open_input_file('/Users/aloneirew/workspace/DenseEREGraph/data/DOT_format/TimeBankDense_test_dot.json')

_output_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/prompt/prompt_nt_gpt4o_2.jsonl'
_response_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/prompt/prompt_nt_gpt4o_2.jsonl'
_final_output_file = '/Users/aloneirew/workspace/DenseEREGraph/data/my_data/prompt/prompt_nt_gpt4o_task_description_2.json'

_num_of_files_to_prepare = -1
_num_of_examples = 1
_selected_file = "21_final.json"
_reduce = -1 # reduction of edges in percentage
_model_id = "gpt-4o"

Step-1: Create the request and save it to a file

In [34]:
print("Running Step-1")
from_dataset_to_batch_req(_test_folder, _train_folder, _dot_train_data, _output_file, _num_of_files_to_prepare, _num_of_examples, _selected_file, _model_id, _reduce, gen_pairs=True)

Running Step-1
Number of tokens in prompt=3062
Number of tokens in prompt=2588
Number of tokens in prompt=2530
Number of tokens in prompt=2503
Number of tokens in prompt=2547
Number of tokens in prompt=2318
Number of tokens in prompt=2422
Number of tokens in prompt=2408
Number of tokens in prompt=2669
Total number of tokens in all prompts=23047


Step-2: upload the request to the server

In [48]:
print("Running Step-2")
print(_output_file)
_input_request_jsonl = _output_file
_batch_input_file_id = upload_batch_request(_client, _input_request_jsonl)

Running Step-2
/Users/aloneirew/workspace/DenseEREGraph/data/my_data/prompt/prompt_nt_gpt4o_2.jsonl
Batch input file with id file-GwXQjq3s6xjcGNts7WJWao uploaded


Step-3: Run the batch

In [49]:
print("Running Step-3")
print(_batch_input_file_id)
_batch_run_id = run_batch(_client, _batch_input_file_id)

Running Step-3
file-GwXQjq3s6xjcGNts7WJWao
Batch object created: Batch(id='batch_678bba8f055c8190ad2dca5d8fa40e05', completion_window='24h', created_at=1737210511, endpoint='/v1/chat/completions', input_file_id='file-GwXQjq3s6xjcGNts7WJWao', object='batch', status='validating', cancelled_at=None, cancelling_at=None, completed_at=None, error_file_id=None, errors=None, expired_at=None, expires_at=1737296911, failed_at=None, finalizing_at=None, in_progress_at=None, metadata={'description': 'eventfull gpt4o batch job'}, output_file_id=None, request_counts=BatchRequestCounts(completed=0, failed=0, total=0))


Step-4: Check the status of the batch

In [53]:
print("Running Step-4")
print(_batch_run_id)
check_batch_status(_client, _batch_run_id, _response_file)

Running Step-4
batch_678bba8f055c8190ad2dca5d8fa40e05
Batch status: Batch(id='batch_678bba8f055c8190ad2dca5d8fa40e05', completion_window='24h', created_at=1737210511, endpoint='/v1/chat/completions', input_file_id='file-GwXQjq3s6xjcGNts7WJWao', object='batch', status='completed', cancelled_at=None, cancelling_at=None, completed_at=1737210829, error_file_id=None, errors=None, expired_at=None, expires_at=1737296911, failed_at=None, finalizing_at=1737210827, in_progress_at=1737210512, metadata={'description': 'eventfull gpt4o batch job'}, output_file_id='file-7S9om8y9TRfuiQd3hbAeML', request_counts=BatchRequestCounts(completed=9, failed=0, total=9))
{"id": "batch_req_678bbbcb9454819080fd74532a985dab", "custom_id": "APW19980227.0494.json", "response": {"status_code": 200, "request_id": "9c0b49585d636d1bbf02ea9a7b76810f", "body": {"id": "chatcmpl-Ar47U5O65cy2dePPfCCnFcRH1znP7", "object": "chat.completion", "created": 1737210584, "model": "gpt-4o-2024-08-06", "choices": [{"index": 0, "messag

Step-5: Convert response to DOT format response

In [55]:
print("Running Step-5")
convert_response(_response_file, _final_output_file)
print("Done!")

Running Step-5
Done!


Cancelling a batch

In [13]:
_client.batches.cancel("batch_67710f830ccc8190ba34189e999d8a82")

ConflictError: Error code: 409 - {'error': {'message': "Cannot cancel a batch with status 'failed'.", 'type': 'invalid_request_error', 'param': None, 'code': None}}